#### 1 Import Functions

**FOOOF/SpecParam**

FOOOF (fitting oscillations & one over f) and its successor SpecParam are spectral parameterization methods that fit a model to the power spectrum of neural signals, separating aperiodic (1/f) and periodic (oscillatory) components. These methods work directly on the PSD and do not require resampling or averaging across multiple epochs.

**Why FOOOF/SpecParam is better for short single-trial epochs:**
- FOOOF/SpecParam can reliably fit the aperiodic exponent on short signals (e.g., 2 seconds) and even single trials, as long as the PSD has enough frequency resolution.
- IRASA (Irregular resampling auto-spectral analysis) requires longer signals because it resamples the input by multiple factors and averages PSDs, which is unreliable or fails for short epochs due to insufficient data after resampling.

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
# from fooof import FOOOF
# from fooof.bands import Bands
import pywt
from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
from scipy.signal import welch
from specparam import SpectralModel
from neurodsp.aperiodic import compute_irasa, fit_irasa

import os
import sys
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# project paths and helpers
sys.path.insert(0, './lib')
sys.path.insert(0, './utils/')
import utils_io
from lib_data import DATA_IO

In [ ]:
PATH_CURR = os.path.abspath(os.curdir)
PATH      = (str(Path(PATH_CURR).parent))
fs        = 2048

#### 2 Helper Functions

In [ ]:
#############################################
# Compute PSD with complex Morlet wavelets
#############################################

def morlet_psd_pywt(x, fs, fmin=1, fmax=90, n_freqs=90):
    """
    Compute power spectrum with complex Morlet wavelet convolution using PyWavelets.
    - x: 1D signal
    - fs: sampling rate
    - fmin/fmax: frequency range of interest
    - n_freqs: number of frequencies
    
    Returns:
        freqs : array of frequencies (Hz)
        psd   : power at each frequency
    """
    freqs = np.linspace(fmin, fmax, n_freqs)
    wavelet = pywt.ContinuousWavelet('cmor1.5-1.0')  # Complex Morlet wavelet with fb=1.5, fc=1.0
    scales = pywt.central_frequency(wavelet) * fs / freqs
    coefs, _ = pywt.cwt(x, scales, wavelet, sampling_period=1/fs)
    power = np.abs(coefs) ** 2
    T = power.shape[1]
    # ~2 wavelet widths at each scale (heuristic; adjust if needed)
    for i, s in enumerate(scales):
        edge = int(np.ceil(2 * s))
        if edge > 0 and 2*edge < T:
            power[i, :edge] = np.nan
            power[i, -edge:] = np.nan
    psd = np.nanmean(power / scales[:, None], axis=1)

    return freqs, psd

def welch_psd(x, fs, fmin=1, fmax=90):
    """
    Compute power spectrum using Welch's method (scipy.signal.welch).
    Returns frequencies and PSD values only for fmin <= freq <= fmax.
    """
    
    if len(x) < fs: # Skip if signal is shorter than nperseg
        return None, None
        # # Round nperseg down to the next power of 2
        # nperseg = 2**int(np.floor(np.log2(len(x))))
        # if nperseg < 256: # Skip if signal is shorter than 256 samples
        #     return None, None
    
    # When nperseg = fs then the frequency steps that the psd is calculated form is 1 Hz resolution
    freqs, psd = welch(x, fs=fs, nperseg=fs)
    mask = (freqs >= fmin) & (freqs <= fmax)
    return freqs[mask], psd[mask]


#############################################
# Linear interpolate across 47–53 Hz
#############################################

def remove_line_noise_gap(freqs, psd, f_lo=47, f_hi=53):
    mask = (freqs < f_lo) | (freqs > f_hi)
    # linear interpolation in the gap
    interp_vals = np.interp(
        freqs[~mask], 
        [f_lo, f_hi], 
        [psd[freqs==f_lo][0], psd[freqs==f_hi][0]]
    )
    psd[~mask] = interp_vals
    return psd

#############################################
# Run Fooof on one PSD
#############################################

# def run_fooof(freqs, psd, f_range=(60,90), mode='fixed'):
#     fm = FOOOF(aperiodic_mode=mode, verbose=False)
#     # Fit the model
#     fm.fit(freqs, psd, f_range)
#     # fm.plot()
#     offset, exponent = fm.aperiodic_params_ # The exponent = -slope
#     fitted_spectrum = fm.fooofed_spectrum_
#     return exponent, offset, fitted_spectrum

def run_specparam(freqs, psd, f_range=(60, 90), mode='fixed'):
    """
    Run SpecParam (successor to FOOOF) on a PSD.

    Parameters
    ----------
    freqs : 1d array
        Frequency vector (Hz).
    psd : 1d array
        Power spectrum (linear scale).
    f_range : tuple
        Frequency range to fit (low, high).
    mode : str
        'fixed' or 'knee' aperiodic mode.

    Returns
    -------
    exponent : float
        The aperiodic exponent (slope of 1/f).
    offset : float
        The aperiodic offset (power at 1 Hz, log10 units).
    fitted_spectrum : 1d array
        The full modelled spectrum across `freqs`.
    """

    sm = SpectralModel(aperiodic_mode=mode, verbose=False)
    sm.fit(freqs, psd, f_range) # Fit to data

    # Extract parameters
    ap_params = sm.get_params('aperiodic_params')
    if mode == 'fixed':
        offset, exponent = ap_params
    elif mode == 'knee':
        offset, knee, exponent = ap_params

    # Get model spectrum
    fitted_spectrum = sm.modeled_spectrum_

    return exponent, offset, fitted_spectrum

def run_irasa_epochs(epochs, fs, f_range=(60, 90)):
    """
    Run IRASA across multiple epochs and return mean aperiodic fit.

    Parameters
    ----------
    epochs : list of 1d arrays
        List of time series (epochs) from the same condition/channel.
    fs : float
        Sampling rate (Hz).
    f_range : tuple
        Frequency range for fitting (Hz).

    Returns
    -------
    exponent : float
        Mean aperiodic exponent across epochs (from averaged PSD).
    offset : float
        Offset from fit to averaged PSD.
    fitted_spectrum : 1d array
        Modeled 1/f spectrum across freqs, from averaged PSD.
    freqs : 1d array
        Frequency vector (Hz).
    """

    all_psd_aper = []

    for x in epochs:
        freqs, psd_aperiodic, psd_periodic = compute_irasa(x, fs, f_range=f_range, hset=np.arange(1.1, 1.5, 0.05))
        all_psd_aper.append(psd_aperiodic)

    # Average across epochs
    mean_psd_aper = np.mean(all_psd_aper, axis=0)

    # Fit slope on mean fractal PSD
    offset, exponent = fit_irasa(freqs, mean_psd_aper)

    # Reconstruct fitted line
    fitted_spectrum = 10 ** (offset - exponent * np.log10(freqs))

    return exponent, offset, fitted_spectrum


#############################################
# Main loop over epochs
#############################################

def compute_individual_exponents(patient, channel, group, trialPhase, fs, f_range, mode='fixed', aperiodicFit='FOOOF'):
    # drop rows with empty or null signals
    potentials = []
    for pot in group[trialPhase]:
        if pot is None:
            continue
        pot = np.asarray(pot)
        if pot.size == 0:
            continue
        potentials.append(pot)
    if len(potentials) == 0:
        return

    if len(potentials) == 0: # Skip empty groups
        return None
    
    # IRASA takes the raw signals to compute the aperiodic psd, thus with multiple epochs we need to input all epochs
    if aperiodicFit == 'IRASA':
        # IRASA needs all epochs to be the same length
        # Trim all epochs to the same length based on the 10th percentile of lengths (keeps ≥90% of trials)
        lengths = np.array([len(pot) for pot in potentials])
        target_len = math.floor(np.percentile(lengths, 10))

        # Filter out too-short epochs and trim all to target_len
        potentials = [pot[:target_len] for pot in potentials if len(pot) >= target_len]
        if len(potentials) == 0:
            return None  
        
        exponent, offset, fitted_spectrum = run_irasa_epochs(potentials, fs, f_range=f_range)
        
        return {
            "patient": patient,
            "channel": channel,
            "exponent": exponent,
            "offset": offset,
            "fitted_spectrum": fitted_spectrum
        }
    # FOOOF/SpecParam takes the psd, thus with multiple epochs we need to average the psd first and then fit FOOOF to that average psd
    elif aperiodicFit == 'FOOOF':
        psds = []
        all_freqs = []
        for pot in potentials:
            if method == 'morlet':
                freqs, psd = morlet_psd_pywt(pot, fs)
            elif method == 'welch':
                freqs, psd = welch_psd(pot, fs)
            else:
                raise ValueError("Invalid method. Choose 'morlet' or 'welch'.")
            if freqs is None or psd is None:
                continue
            if freqs is not None:
                all_freqs.append(freqs)
            # if both f_range values are smaller than 47 Hz or bigger than 53 Hz, skip this step
            if f_range[1] < 47 or f_range[0] > 53:
                pass
            else:
                psd = remove_line_noise_gap(freqs, psd)
            psds.append(psd)
        # Check if all frequency arrays are the same
        if not all(np.array_equal(all_freqs[0], f) for f in all_freqs):
            raise ValueError("Frequency arrays are not the same across all PSDs.")
        if len(psds) == 0:
            return None
        psds = np.array(psds)
        mean_psd = np.nanmean(psds, axis=0)
        exponent, offset, fitted_spectrum = run_specparam(all_freqs[0], mean_psd, f_range=f_range, mode=mode)
        return {
            "patient": patient,
            "channel": channel,
            "exponent": exponent,
            "offset": offset,
            "fitted_spectrum": fitted_spectrum
        }

def compute_exponents_for_df(df, fs, trialPhase, f_range=(60,90), mode='fixed', aperiodicFit='FOOOF'):
    channel_exponents = {}
    if df is None or len(df)==0:
        return channel_exponents 
    
    # Check if "ECoG_channel" column exists, if not check if "LFP_channel" exists
    if "ECoG_channel" in df.columns:
        channelName = "ECoG_channel"
    elif "LFP_channel" in df.columns:
        channelName = "LFP_channel"
    else:
        raise ValueError("DataFrame must contain 'ECoG_channel' or 'LFP_channel' column.")
    
    # For each patient and channel calculate psd and then average that psd across epochs
    channel_exponents = []
    # groups = list(df.groupby(["patient", channelName]))
    groups = [(keys, grp) for keys, grp in df.groupby(["patient",channelName]) if len(grp) > 0]

    if parallelComputing:
        # Fit Fooof to each group in parallel with a progress bar
        with tqdm_joblib(desc="Fooof fits per patient and channel", total=len(groups)) as progress_bar:
            results = Parallel(n_jobs=-1)(
                delayed(compute_individual_exponents)(patient, channel, group, trialPhase, fs, f_range, mode, aperiodicFit=aperiodicFit)
                for (patient, channel), group in groups
            )
        # Remove all None results
        results = [res for res in results if res is not None]
        channel_exponents.extend(results)
    else:
        for (patient, channel), group in tqdm(groups, desc="Fooof fits per patient and channel", total=len(groups)):
            res = compute_individual_exponents(patient, channel, group, trialPhase, fs, f_range, mode, aperiodicFit=aperiodicFit)
            if res is not None:
                channel_exponents.append(res)

    return channel_exponents

In [ ]:
# # Plot frequency spectrum and psd
# plt.figure(figsize=(10, 6))
# plt.semilogy(freqs, psd, label='PSD', color='blue')
# plt.xlim([1, 100])
# plt.ylim([1e-6, 1e1])
# plt.xlabel('Frequency (Hz)')
# plt.ylabel('Power Spectral Density (µV²/Hz)')
# plt.title('Power Spectral Density with Morlet Wavelets')
# plt.grid()
# plt.legend()
# plt.show(block=True)

# # Plot raw oscillatory signal
# plt.figure()
# plt.plot(x)
# plt.title("Raw input signal")
# plt.show(block=True)

#### 3 Load event files

In [ ]:
EVENTS_ECOG      = utils_io.load_ECoG_events(event_category="tapping", fs=fs)
EVENTS_ECOG_LID  = EVENTS_ECOG['controlateral']['LID']
EVENTS_ECOG_noLID= EVENTS_ECOG['controlateral']['noLID']
# Given the event starting times, split the noLID dataframe into a medOff and medOn dataframe 
# as defined by events that occur before or after 30 minutes after medication intake.
EVENTS_ECOG_noLID_medOff = EVENTS_ECOG_noLID[EVENTS_ECOG_noLID.event_start_time <= 30]
EVENTS_ECOG_noLID_medOn  = EVENTS_ECOG_noLID[EVENTS_ECOG_noLID.event_start_time > 30]

EVENTS_LFP             = utils_io.load_LFP_events(event_category="tapping", stn_areas=["motor"], fs=fs)
EVENTS_LFP_MOTOR_LID   = EVENTS_LFP['motor']['controlateral']['LID']
EVENTS_LFP_MOTOR_noLID = EVENTS_LFP['motor']['controlateral']['noLID']
# Given the event starting times, split the noLID dataframe into a medOff and medOn dataframe 
# as defined by events that occur before or after 30 minutes after medication intake.
EVENTS_LFP_MOTOR_noLID_medOff = EVENTS_LFP_MOTOR_noLID[EVENTS_LFP_MOTOR_noLID.event_start_time <= 30]
EVENTS_LFP_MOTOR_noLID_medOn  = EVENTS_LFP_MOTOR_noLID[EVENTS_LFP_MOTOR_noLID.event_start_time > 30]

# Delete large variables that are not used to save memory
# del EVENTS_ECOG, EVENTS_LFP, EVENTS_ECOG_noLID, EVENTS_LFP_MOTOR_noLID

In [ ]:
# filepath = "c:\\Users\\filip\\OneDrive - Charité - Universitätsmedizin Berlin\\Kühn Lab\\Projects\\dysk_ecoglfp\\data\\events\\aperiodic_Component\\FOOOF_exponents_ECoG_noLID_medOff_preEvent.pkl"
# df_noLID_medOff_preEvent = pd.read_pickle(filepath)
# # Plot the fitted spectrum for the first row as an example

# fitted_spectrum = df_noLID_medOff_preEvent['fitted_spectrum'].iloc[0]
# fitted_spectrum_linear = 10 ** fitted_spectrum
# freqs = np.linspace(60, 90, len(fitted_spectrum_linear))  # match frequency range to fitted spectrum length

# plt.figure(figsize=(10, 6))
# # plot log-log
# plt.loglog(freqs, fitted_spectrum_linear, label='Fitted Spectrum', color='red')
# # plt.semilogy(freqs, fitted_spectrum_linear, label='Fitted Spectrum', color='red') 
# plt.xlabel('Frequency (Hz)')
# plt.ylabel('Power')
# plt.title('Example FOOOF Fit for noLID_medOff_preEvent')
# plt.show()

In [ ]:
method = 'welch'  # 'morlet' or 'welch'
parallelComputing = False
f_range = (60,90) # Range to compute psd 
aperiodicFit = 'FOOOF' # 'FOOOF' or 'IRASA'
if aperiodicFit=='IRASA':
    method = 'welch'  # IRASA uses welch method for psd computation

trialPhase_list = ["pre_event_recording", "event_recording", "post_event_recording"]
modalities = ["ECoG", "LFP"]
conds = ["noLID_medOff", "noLID_medOn", "LID"]

# Store matrices for each modality
modality_matrices = {}
for mod in modalities:
    df_matrix = pd.DataFrame(index=[tp.replace("pre_event_recording", "preEvent")
                                    .replace("post_event_recording", "postEvent")
                                    .replace("event_recording", "event")
                                    for tp in trialPhase_list], columns=conds)
    for trialPhase in trialPhase_list:
        phaseString = trialPhase.replace("pre_event_recording", "preEvent") \
                                .replace("post_event_recording", "postEvent") \
                                .replace("event_recording", "event")
        groups = [
            ("ECoG", "noLID_medOff", EVENTS_ECOG_noLID_medOff),
            ("ECoG", "noLID_medOn",  EVENTS_ECOG_noLID_medOn),
            ("ECoG", "LID",          EVENTS_ECOG_LID),
            ("LFP",  "noLID_medOff", EVENTS_LFP_MOTOR_noLID_medOff),
            ("LFP",  "noLID_medOn",  EVENTS_LFP_MOTOR_noLID_medOn),
            ("LFP",  "LID",          EVENTS_LFP_MOTOR_LID),
        ]
        for cond in conds:
            # Find the correct df for this modality and condition
            df = None
            for m, c, d in groups:
                if m == mod and c == cond:
                    df = d
                    break
            folderPath = DATA_IO.path_events + "/aperiodic_Component/"
            if not os.path.exists(folderPath):
                os.makedirs(folderPath)
            filepath = folderPath + f"/{aperiodicFit}_exponents_{mod}_{cond}_{phaseString}_{method}.pkl"
            # Check if it already exists
            if os.path.exists(filepath):
                print(f"File {filepath} already exists, skipping computation.")
                loaded = pd.read_pickle(filepath)
                df_matrix.at[phaseString, cond] = loaded
                continue
            print(f"Processing {mod} | {cond} | {phaseString} | {len(df)} epochs")
            exponents = compute_exponents_for_df(df, fs=fs, trialPhase=trialPhase, f_range=f_range, mode='fixed', aperiodicFit=aperiodicFit)
            df_matrix.at[phaseString, cond] = pd.DataFrame(exponents)
    
    # Save the big DataFrame of DataFrames for this modality
    f_range_str = '_'.join(str(x) for x in f_range)
    savepath = DATA_IO.path_events + f"/aperiodic_Component/{aperiodicFit}_exponents_{mod}_{method}_{'-'.join(str(x) for x in f_range)}Hz.pkl"
    df_matrix.to_pickle(savepath)
    print(f"Saved matrix for {mod} to {savepath}")
    modality_matrices[mod] = df_matrix
    print(f"---------------------------------------------------------------------------------------------------------")

#### 4 Plot results

In [ ]:
#############################################
# Boxplots
#############################################
'''
A steeper (more negative) aperiodic slope (i.e., bigger exponent) indicates relatively more low-frequency power and 
    less high-frequency power. This is often associated with increased inhibition or reduced excitation.
A flatter (less negative) slope (i.e., smaller exponent) suggests relatively more high-frequency power, 
    which is often interpreted as increased excitation or reduced inhibition.
'''

phaseString = "event" # Can be "preEvent", "event", or "postEvent"
methodPlot = 'welch'  # 'morlet' or 'welch'
f_range = (5,45) # Range to compute psd 

fig, ax = plt.subplots(figsize=(9,5))
conds = ["noLID_medOff","noLID_medOn","LID"]
modalities = ["ECoG","LFP"]
positions=[]; boxdata=[]; labels=[]
pos=1
for mod in modalities:
    for cond in conds:
        # Load the pickle file
        filepath = DATA_IO.path_events + f"/aperiodic_Component/FOOOF_exponents_{mod}_{methodPlot}_{'-'.join(str(x) for x in f_range)}Hz.pkl"
        # See if the file exists
        if not os.path.exists(filepath):
            print(f"File {filepath} does not exist, skipping.")
            continue
        df_exponents = pd.read_pickle(filepath)
        # Access the DataFrame for the specified phaseString and condition
        df_exponents = df_exponents.at[phaseString, cond]
        if df_exponents is None or len(df_exponents) == 0:
            print(f"No data for {mod} | {cond} | {phaseString}, skipping.")
            continue
        # Extract exponent values
        vals = df_exponents['exponent'].values
        boxdata.append(vals)
        positions.append(pos)
        labels.append(f"{mod}\n{cond}")
        pos+=1
    pos+=1  # gap
ax.boxplot(boxdata, positions=positions, widths=0.6)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel("Aperiodic exponent (Fooof)")
ax.set_title("Fooof-derived exponents")
plt.tight_layout()
plt.show()
# Save plot 
saveFile = DATA_IO.path_events + f"/aperiodic_Component/Figures/FOOOF_exponents_boxplot_{phaseString}_{methodPlot}_{'-'.join(str(x) for x in f_range)}Hz.svg"
fig.savefig(saveFile, dpi=300)

In [ ]:
from scipy.stats import ranksums
modality = "ECoG" # "ECoG" or "LFP"
phaseString = "event" # Can be "preEvent", "event", or "postEvent"
methodPlot = 'welch'  # 'morlet' or 'welch'
f_range = (5,45) # Range to compute psd 

# Load the pickle file
filepath = DATA_IO.path_events + f"/aperiodic_Component/FOOOF_exponents_{modality}_{methodPlot}_{'-'.join(str(x) for x in f_range)}Hz.pkl"
df_exponents = pd.read_pickle(filepath)
# Run a rank sum test on noLID_medOff and noLID_medOn exponents
p = ranksums(df_exponents.at[phaseString, 'noLID_medOff']['exponent'].values,
              df_exponents.at[phaseString, 'noLID_medOn']['exponent'].values)
print(f"Rank sum test p-value between noLID_medOff and noLID_medOn exponents: {p.pvalue}")

p = ranksums(df_exponents.at[phaseString, 'noLID_medOff']['exponent'].values,
              df_exponents.at[phaseString, 'LID']['exponent'].values)
print(f"Rank sum test p-value between noLID_medOff and LID exponents: {p.pvalue}")

p = ranksums(df_exponents.at[phaseString, 'noLID_medOn']['exponent'].values,
              df_exponents.at[phaseString, 'LID']['exponent'].values)
print(f"Rank sum test p-value between noLID_medOn and LID exponents: {p.pvalue}")